# YouTube Comment Scraper â€” Topik Listrik
Pipeline: **Scraping â†’ Cleaning â†’ Export CSV**

Tujuan: mengumpulkan komentar YouTube untuk labeling manual & fine-tuning IndoBERT.

## 0. Install Dependensi

In [ ]:
!pip install -q youtube-comment-downloader pandas requests

   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 300.4/300.4 kB 23.9 MB/s eta 0:00:00


## 1. Import & Konfigurasi

In [ ]:
import csv, time, os, re
import unicodedata
import requests
import pandas as pd
from datetime import datetime
from youtube_comment_downloader import YoutubeCommentDownloader

VIDEO_URLS = [
    "https://www.youtube.com/watch?v=re-gdNPxUwU",
    "https://www.youtube.com/watch?v=NcTgBr9JBAo",
    "https://www.youtube.com/watch?v=08Qr6xL-xa8",
    "https://www.youtube.com/watch?v=H24JtTk831Y",
    "https://www.youtube.com/watch?v=q2nDnDABeiI",
    "https://www.youtube.com/watch?v=ULLgvYxorew",
    "https://www.youtube.com/watch?v=IahX_99vclU",
    "https://www.youtube.com/watch?v=A9iCTyQL4ek",
    "https://www.youtube.com/watch?v=Vfex7GADWHg",
    "https://www.youtube.com/watch?v=C3OuukvuTao",
    "https://www.youtube.com/watch?v=LDtaEN5FFgY",
    "https://www.youtube.com/watch?v=UESzjnrkXmo",
    "https://www.youtube.com/watch?v=bC7YjmUOnAU",
    "https://www.youtube.com/watch?v=91MVR78A6vw",
    "https://www.youtube.com/watch?v=D8qnpCAaDqs",
]

RAW_FILE     = "data/raw/hasil_komentar_youtube.csv"
CLEAN_FILE   = "data/processed/hasil_komentar_bersih.csv"
MAX_COMMENTS = 3000
DELAY_SEC    = 2

FIELDNAMES = ["channel", "video_title", "video_id", "comment", "author", "likes", "time"]
HEADERS    = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

## 2. Fungsi Pembantu

In [ ]:
def extract_video_id(url):
    """Ekstrak video ID dari URL YouTube."""
    m = re.search(r"(?:v=|youtu\.be/)([A-Za-z0-9_-]{11})", url)
    return m.group(1) if m else "unknown"


def get_video_info(url):
    """Ambil judul dan nama channel dari URL video."""
    try:
        html      = requests.get(url, headers=HEADERS, timeout=15).text
        title_m   = re.search(r'"title":"([^"]+)"', html)
        channel_m = re.search(r'"ownerChannelName":"([^"]+)"', html) \
                 or re.search(r'"author":"([^"]+)"', html)
        title   = title_m.group(1)   if title_m   else "Unknown Title"
        channel = channel_m.group(1) if channel_m else "Unknown Channel"
        if "\\u" in title:
            title = title.encode("utf-8").decode("unicode_escape")
    except Exception as e:
        print(f"  [!] Gagal ambil info: {e}")
        title, channel = "Unknown Title", "Unknown Channel"
    return title, channel


def save_csv(rows, filepath):
    """Simpan list of dict ke file CSV."""
    with open(filepath, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(rows)


# â”€â”€ Deteksi Noise â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
NOISE_KATA = {"pertamax", "first", "pertama"}

def is_noise(text):
    """Return True jika komentar dianggap tidak informatif."""
    t = str(text).strip()
    if not t:
        return True
    # hanya URL / mention / hashtag / angka / simbol
    if re.match(r'^(https?://\S+|@\w+|#\w+|\d+|[^\w\s]+)$', t):
        return True
    # hanya emoji
    tanpa_spasi = t.replace(" ", "")
    if tanpa_spasi and all(unicodedata.category(c) in ("So", "Cs", "Mn") for c in tanpa_spasi):
        return True
    # kata tunggal yang termasuk noise
    if len(t.split()) == 1 and t.lower() in NOISE_KATA:
        return True
    return False

## 3. Scraping Komentar

In [ ]:
def scrape_video(url, index, total):
    """Scrape komentar dari satu video YouTube."""
    video_id       = extract_video_id(url)
    title, channel = get_video_info(url)
    short_title    = title[:65] + ("..." if len(title) > 65 else "")
    print(f"[{index}/{total}] {channel} â€” {short_title}")

    rows, count = [], 0
    try:
        for cmt in YoutubeCommentDownloader().get_comments_from_url(url, sort_by=0):
            rows.append({
                "channel":     channel,
                "video_title": title,
                "video_id":    video_id,
                "comment":     cmt.get("text", "").strip(),
                "author":      cmt.get("author", "").strip(),
                "likes":       cmt.get("votes", 0) or 0,
                "time":        cmt.get("time_parsed", "") or cmt.get("time", ""),
            })
            count += 1
            if count % 200 == 0:
                print(f"  ... {count} komentar")
            if MAX_COMMENTS and count >= MAX_COMMENTS:
                break
    except Exception as e:
        print(f"  [!] Error: {e}")

    print(f"  -> {count} komentar terkumpul")
    return rows


def scrape_all():
    """Scrape semua video dan simpan ke CSV secara inkremental."""
    print(f"Scraping {len(VIDEO_URLS)} video | {datetime.now():%Y-%m-%d %H:%M:%S}")
    print("-" * 60)

    all_rows, failed = [], []
    for i, url in enumerate(VIDEO_URLS, 1):
        rows = scrape_video(url, i, len(VIDEO_URLS))
        if rows:
            all_rows.extend(rows)
            save_csv(all_rows, RAW_FILE)      # simpan inkremental
        else:
            failed.append(url)
        if i < len(VIDEO_URLS):
            time.sleep(DELAY_SEC)

    print("-" * 60)
    print(f"Selesai. Total: {len(all_rows)} komentar | Gagal: {len(failed)} video")
    print(f"Raw CSV  : {os.path.abspath(RAW_FILE)}")
    if failed:
        print("Video gagal:")
        for u in failed:
            print(f"  - {u}")
    return all_rows

## 4. Cleaning Data
> Preprocessing NLP lanjutan (tokenizing, stemming, stopword removal) dilakukan pada tahap berikutnya.

In [ ]:
def clean_data(filepath):
    """Baca CSV raw, bersihkan, dan kembalikan DataFrame."""
    df = pd.read_csv(filepath, encoding="utf-8-sig")

    df = df[["author", "comment"]].copy()
    df["author"]  = df["author"].astype(str)
    df["comment"] = df["comment"].astype(str)

    df = df.dropna(subset=["comment"])
    df = df[~df["comment"].apply(is_noise)]
    df = df[df["comment"].str.strip() != ""]
    df = df[df["comment"].str.len() >= 3]
    df = df.drop_duplicates(subset=["comment"], keep="first")
    df = df.reset_index(drop=True)

    df.to_csv(CLEAN_FILE, index=False, encoding="utf-8-sig")
    print(f"Clean CSV: {os.path.abspath(CLEAN_FILE)}")
    print(f"Jumlah komentar bersih: {len(df)}")
    return df

## 5. Jalankan Pipeline

In [ ]:
# Scraping
scrape_all()

# Cleaning
df_clean = clean_data(RAW_FILE)
df_clean.head(10)

Scraping 15 video | 2026-05-24 06:20:13
------------------------------------------------------------
[1/15] KOMPASTV â€” Menkeu Purbaya Jawab Isu Pajak Kendaraan Listrik Naik: Tak Ada Pe...
  -> 88 komentar terkumpul
[2/15] METRO TV â€” Aturan Baru Pajak Kendaraan Listrik Resmi Berlaku, Ini Kata Masya...
  ... 200 komentar
  -> 354 komentar terkumpul
[3/15] METRO TV â€” Pemilik Kendaraan EV Pertanyakan Penerapan Pajak - [Newsline Bisn...
  ... 200 komentar
  ... 400 komentar
  ... 600 komentar
  ... 800 komentar
  ... 1000 komentar
  -> 1177 komentar terkumpul
[4/15] METRO TV â€” Skema Baru Pembayaran Pajak Kendaraan Listrik - [Metro Siang]
  ... 200 komentar
  ... 400 komentar
  -> 427 komentar terkumpul
[5/15] CNN Indonesia â€” Pajak Mobil Listrik Tak Lagi Sepenuhnya Gratis
  -> 13 komentar terkumpul
[6/15] Official iNews â€” HEBOH! Kenaikan Pajak Mobil Listrik, Begini Respons Menkeu Purbay...
  -> 108 komentar terkumpul
[7/15] ANTARA News â€” Mulai 2026, Kendaraan Listrik dikenakan 

,author,comment
0,@binjak4719,"Pemerintah panic cari masukkan , satu2 ny yg m..."
1,@Lanternguy95,Ah purbaya ngomong doang sma aja ternyata kya ...
2,@budilewiyanto7798,Yg jd viral pajak apaan yg dijawab lain ! ðŸ˜‚ðŸ˜‚ðŸ˜‚
3,@SoulLantern13,"Awalnya dikasih diskon biar pada pindah,pas se..."
4,@herisusanto101,Dulunya gembar gembor bigini begono gak taunya...
5,@TahuLebihJadiPintar,Purbaya jadi bingung harus ngomong apa ðŸ˜‚
6,@Jemmy_BengkelKu,Owh bener juga sih lok merk mobil ga patuhi at...
7,@amift7555,semoga pak pur selalu diberikan kesehatan dan ...
8,@RahmatNurjaman-k6p,Pak purbaya kayaknya hidupnya madih bahagia se...
9,@AmarMaulana-tf9gd,Komponen perhitungan pajak kepemilikan tahunan...
